In [ ]:
#| default_exp misc.conv_decomposer

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

The `Conv_Decomposer` class reduces model size and FLOPs by factorizing Conv2d layers into three smaller convolutions using Tucker decomposition. This is the Conv2d counterpart of `FC_Decomposer` (which uses SVD for Linear layers).

**How it works:** A Conv2d weight `[C_out, C_in, H, W]` is decomposed into:
1. `Conv2d(C_in, R_in, 1)` — pointwise input channel compression
2. `Conv2d(R_in, R_out, (H, W))` — spatial convolution at reduced rank
3. `Conv2d(R_out, C_out, 1)` — pointwise output channel expansion

### When to Use

| Scenario | Recommendation |
|----------|----------------|
| Large 3x3 or larger convolutions | **Highly recommended** — significant FLOP savings |
| 1x1 pointwise convolutions | Skipped automatically (already minimal) |
| Depthwise / grouped convolutions | Skipped (Tucker assumes standard convolution) |
| First layer (C_in=3) | Works but limited benefit |
| Post-training compression | Fine-tune after decomposition for best accuracy |

In [ ]:
#| export
import torch
import torch.nn as nn
import copy

In [ ]:
#| export
def _unfold(tensor, mode):
    "Unfold a tensor along a mode into a matrix"
    return tensor.moveaxis(mode, 0).flatten(1)

def _partial_tucker(weight, ranks, n_iter=5):
    "Partial Tucker decomposition on modes [0, 1] via alternating SVD (HOOI)"
    # Initialize factors from SVD of mode unfoldings
    U0 = torch.linalg.svd(_unfold(weight, 0), full_matrices=False)[0][:, :ranks[0]]
    U1 = torch.linalg.svd(_unfold(weight, 1), full_matrices=False)[0][:, :ranks[1]]

    for _ in range(n_iter):
        # Project out mode 0 using U0, then update U1
        proj = torch.einsum('oihw, or -> rihw', weight, U0)
        U1 = torch.linalg.svd(_unfold(proj, 1), full_matrices=False)[0][:, :ranks[1]]
        # Project out mode 1 using U1, then update U0
        proj = torch.einsum('oihw, is -> oshw', weight, U1)
        U0 = torch.linalg.svd(_unfold(proj, 0), full_matrices=False)[0][:, :ranks[0]]

    # Core = W ×₀ U0ᵀ ×₁ U1ᵀ
    core = torch.einsum('oihw, or, is -> rshw', weight, U0, U1)
    return core, [U0, U1]


class Conv_Decomposer:
    "Decompose Conv2d layers using Tucker decomposition to reduce parameters and FLOPs"

    def __init__(self): pass

    def decompose(self,
                  model: nn.Module,              # The model to decompose
                  percent_removed: float = 0.5,  # Fraction of rank to remove per mode [0, 1)
    ) -> nn.Module:
        "Recursively decompose all eligible Conv2d layers in the model"
        if not (0 <= percent_removed < 1):
            raise ValueError(f"percent_removed must be in range [0, 1), got {percent_removed}")

        new_model = copy.deepcopy(model)
        for name in list(new_model._modules):
            module = new_model._modules[name]
            if len(list(module._modules)) > 0:
                new_model._modules[name] = self.decompose(module, percent_removed)
            elif isinstance(module, nn.Conv2d) and module.groups == 1 and min(module.kernel_size) > 1:
                new_model._modules[name] = self.Tucker(module, percent_removed)
        return new_model

    def Tucker(self,
               layer: nn.Conv2d,          # The Conv2d layer to decompose
               percent_removed: float,    # Fraction of rank to remove per mode
    ) -> nn.Sequential:
        "Perform Tucker decomposition on a single Conv2d layer"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]

        R_out = max(1, int((1 - percent_removed) * C_out))
        R_in  = max(1, int((1 - percent_removed) * C_in))

        core, (U_out, U_in) = _partial_tucker(W, [R_out, R_in])
        # core: (R_out, R_in, H, W), U_out: (C_out, R_out), U_in: (C_in, R_in)

        # 1. Pointwise input compression: (C_in → R_in)
        first = nn.Conv2d(C_in, R_in, 1, bias=False)
        first.weight.data = U_in.t().unsqueeze(-1).unsqueeze(-1)

        # 2. Spatial convolution at reduced rank: (R_in → R_out)
        middle = nn.Conv2d(R_in, R_out, layer.kernel_size, stride=layer.stride,
                           padding=layer.padding, dilation=layer.dilation, bias=False)
        middle.weight.data = core

        # 3. Pointwise output expansion: (R_out → C_out)
        last = nn.Conv2d(R_out, C_out, 1, bias=layer.bias is not None)
        last.weight.data = U_out.unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            last.bias.data = layer.bias.data

        return nn.Sequential(first, middle, last)

In [ ]:
show_doc(Conv_Decomposer.decompose)

---

## Usage Example

```python
from fasterai.misc.conv_decomposer import Conv_Decomposer
from torchvision.models import resnet18

model = resnet18(pretrained=True)
decomposer = Conv_Decomposer()
compressed = decomposer.decompose(model, percent_removed=0.5)

# Check parameter reduction
orig = sum(p.numel() for p in model.parameters())
comp = sum(p.numel() for p in compressed.parameters())
print(f"Compression: {orig/comp:.2f}x")
```

> **Note:** Tucker decomposition uses an iterative algorithm (HOOI), so even at `percent_removed=0.0` there will be small reconstruction error. Fine-tuning after decomposition is recommended.

In [ ]:
#| hide
from fastcore.test import *

decomposer = Conv_Decomposer()

# --- Output shape preserved ---
_m = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
_x = torch.randn(2, 3, 8, 8)
_m_dec = decomposer.decompose(_m, percent_removed=0.5)
test_eq(_m(_x).shape, _m_dec(_x).shape)

# --- percent_removed=0.0 → close reconstruction (HOOI is iterative, not exact) ---
_m2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
_x2 = torch.randn(2, 16, 8, 8)
_m2_dec = decomposer.decompose(_m2, percent_removed=0.0)
test_close(_m2(_x2), _m2_dec(_x2), eps=0.01)

# --- Decomposed structure: Conv2d becomes Sequential of 3 Conv2ds ---
assert isinstance(_m_dec[0], nn.Sequential)
test_eq(len(_m_dec[0]), 3)
test_eq(_m_dec[0][0].kernel_size, (1, 1))  # pointwise in
test_eq(_m_dec[0][1].kernel_size, (3, 3))  # spatial
test_eq(_m_dec[0][2].kernel_size, (1, 1))  # pointwise out

# --- 1x1 convolutions are skipped ---
_m_pw = nn.Sequential(nn.Conv2d(16, 32, 1))
_m_pw_dec = decomposer.decompose(_m_pw, percent_removed=0.5)
assert isinstance(_m_pw_dec[0], nn.Conv2d)  # unchanged, not Sequential

# --- Grouped convolutions are skipped ---
_m_dw = nn.Sequential(nn.Conv2d(16, 16, 3, padding=1, groups=16))
_m_dw_dec = decomposer.decompose(_m_dw, percent_removed=0.5)
assert isinstance(_m_dw_dec[0], nn.Conv2d)  # unchanged

# --- Minimum rank >= 1 even at extreme removal ---
_m3 = nn.Sequential(nn.Conv2d(4, 8, 3, padding=1))
_m3_dec = decomposer.decompose(_m3, percent_removed=0.95)
test_eq(_m3_dec[0][0].out_features if hasattr(_m3_dec[0][0], 'out_features') else _m3_dec[0][0].out_channels, max(1, int(0.05 * 4)))

# --- Bias handling: original bias → last layer gets it ---
_conv_bias = nn.Conv2d(16, 32, 3, padding=1, bias=True)
_dec_bias = decomposer.Tucker(_conv_bias, 0.5)
assert _dec_bias[0].bias is None   # first: no bias
assert _dec_bias[1].bias is None   # middle: no bias
assert _dec_bias[2].bias is not None  # last: has bias

_conv_nobias = nn.Conv2d(16, 32, 3, padding=1, bias=False)
_dec_nobias = decomposer.Tucker(_conv_nobias, 0.5)
assert _dec_nobias[2].bias is None  # last: no bias

# --- Stride/padding transfer to middle conv only ---
_conv_stride = nn.Conv2d(16, 32, 3, stride=2, padding=1)
_dec_stride = decomposer.Tucker(_conv_stride, 0.5)
test_eq(_dec_stride[0].stride, (1, 1))  # pointwise: default
test_eq(_dec_stride[1].stride, (2, 2))  # middle: from original
test_eq(_dec_stride[2].stride, (1, 1))  # pointwise: default

# --- Validation ---
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), percent_removed=1.0)
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), percent_removed=-0.1)

In [ ]:
#| hide
#| slow
from torchvision.models import resnet18

# Decompose a real ResNet-18, verify it still works
_resnet = resnet18(weights=None)
_resnet.eval()
_x = torch.randn(2, 3, 64, 64)
_out_orig = _resnet(_x)

_dec = Conv_Decomposer()
_resnet_dec = _dec.decompose(_resnet, percent_removed=0.5)
_resnet_dec.eval()
_out_dec = _resnet_dec(_x)

# Same output shape
test_eq(_out_orig.shape, _out_dec.shape)

# Outputs are finite (no NaN/Inf)
assert torch.isfinite(_out_dec).all(), "Decomposed ResNet produced non-finite outputs"

# Parameter count reduced
_orig_params = sum(p.numel() for p in _resnet.parameters())
_dec_params = sum(p.numel() for p in _resnet_dec.parameters())
assert _dec_params < _orig_params, f"Expected fewer params: {_dec_params} >= {_orig_params}"
print(f"ResNet-18: {_orig_params:,} → {_dec_params:,} params ({_orig_params/_dec_params:.2f}x compression)")

---

## See Also

- [FC Decomposer](fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm into preceding Conv/Linear layers
- [Pruner](../prune/pruner.html) - Structured pruning that removes entire filters